# 01 · Scaffold and code tour

How the two agents are built: what `agents-cli` generated, and the ADK patterns in the agent code. Nothing in
this notebook changes your project; it reads files from the repo.

In [ ]:
from nbsetup import *

## Scaffolding with agents-cli

`agents-cli scaffold create` generates a production-shaped project in one command: the ADK app, a FastAPI
server (ADK API, A2A and playground routes), Terraform, evaluation boilerplate and tests. Both projects here were
created like this, then the agent code was written into `app/`:

```bash
agents-cli scaffold create growth-risk-agent -a adk -d agent_runtime --cicd-runner skip \
  --bq-analytics --region europe-west3 --root-agent-name growth_risk_orchestrator
agents-cli scaffold create compliance-agent -a adk -d cloud_run --cicd-runner skip \
  --bq-analytics --region europe-west3 --root-agent-name compliance_agent
```

The choices are recorded in `agents-cli-manifest.yaml`.

In [ ]:
%%bash
cd "$REPO/growth-risk-agent"
sed -n '/create_params/,$p' agents-cli-manifest.yaml
echo; find app deployment tests -maxdepth 2 -not -path "*/__pycache__*" -not -path "*/.terraform*" | sort

## The root agent and the App

A `SequentialAgent` runs its sub-agents in a fixed order: no LLM decides the order. You get deterministic
orchestration where you want it, and LLMs inside each step.

In [ ]:
show("growth-risk-agent/app/agent.py", "root_agent = SequentialAgent(")

In [ ]:
show("growth-risk-agent/app/agent.py", "app = App(")

**What to notice**

- `App` wraps the agent with plugins (BigQuery Agent Analytics) and `ResumabilityConfig`.
- Human-in-the-loop needs resumability: after approval, the run continues exactly where it paused instead of starting again.

## Triage: structured output and a code-level guardrail

The first agent turns free text into fields the rest of the pipeline can rely on.

In [ ]:
show("growth-risk-agent/app/schemas.py", "class TriageContext(BaseModel):")

In [ ]:
show("growth-risk-agent/app/sub_agents/triage.py", "triage_agent = Agent(")

In [ ]:
show("growth-risk-agent/app/tools/callbacks.py", "def block_suspicious_input(")

**What to notice**

- `output_schema=TriageContext` forces JSON that matches a Pydantic model; `output_key` stores it in session state, where later agents read it as `{triage_context}`.
- `before_model_callback` runs *before* the LLM is called, so a prompt injection never reaches the model. This is defense in depth: code first, then the model's instructions, then Model Armor (notebook 08).

## Workflow agents: parallel fan-out and a retry loop

In [ ]:
show("growth-risk-agent/app/sub_agents/investigation.py", "parallel_investigation = ParallelAgent(")

In [ ]:
show("growth-risk-agent/app/sub_agents/metrics.py", "sql_refinement_loop = LoopAgent(")

In [ ]:
show("growth-risk-agent/app/tools/callbacks.py", "def gate_investigation(")

**What to notice**

- `ParallelAgent` runs four investigations at the same time; each writes its own `output_key`.
- `LoopAgent`: generate SQL, run it, and on an error let the generator see the error and fix the query. `exit_loop` ends the loop on success; `max_iterations` caps the cost.
- `gate_investigation` (a `before_agent_callback`) skips the whole fan-out when triage asked a clarifying question: returning content from the callback replaces the agent's run.

## Tools are Python functions

ADK turns a function's signature and docstring into the tool declaration the model sees, so the docstring
*is* the prompt for when and how to call the tool.

In [ ]:
show("growth-risk-agent/app/tools/bigquery_tools.py", "def execute_metrics_query(")

**What to notice**

- Read-only by construction: `execute_metrics_query` rejects anything that isn't a `SELECT`.

## Synthesis, memory and artifacts

In [ ]:
show("growth-risk-agent/app/sub_agents/synthesizer.py", "synthesizer_agent = Agent(")

In [ ]:
show("growth-risk-agent/app/tools/memory_tools.py", "async def add_investigation_to_memory(", lines=12)

**What to notice**

- `preload_memory` adds relevant Memory Bank memories to the prompt automatically: that is how a new session recognises a recurring issue.
- The synthesizer must end with exactly one "Primary recommended action". With several, the compliance step picked one arbitrarily (an evaluation found this).
- An `after_agent_callback` saves the report as a downloadable artifact in Cloud Storage.

## The action pipeline: A2A, MCP and human-in-the-loop

In [ ]:
show("growth-risk-agent/app/sub_agents/action_pipeline.py", "compliance_agent = CompactRemoteA2aAgent(")

In [ ]:
show("growth-risk-agent/app/sub_agents/action_pipeline.py", "remediation_agent = Agent(")

**What to notice**

- `RemoteA2aAgent` makes a remote agent look like a local sub-agent: it reads the remote **Agent Card**, then speaks A2A JSON-RPC to it.
- `McpToolset(..., require_confirmation=True)`: the model can *propose* the write, but ADK pauses and asks a human before executing it.
- Both remote services are private Cloud Run services; `gcp_auth.py` impersonates the caller service account to mint an ID token for each one.

## The Compliance Agent

In [ ]:
show("compliance-agent/app/agent.py", "root_agent = Agent(")

In [ ]:
show("compliance-agent/app/tools/action_tools.py", "def check_action_compliance(")

**What to notice**

- The verdict comes from code (`check_action_compliance`), not from the model's judgement; unknown actions fail closed.
- `action_tools.py` is duplicated in both projects (they share no package); a unit test fails if the copies drift.

Next: [02 · Run locally](02_run_locally.ipynb).